# LangChain

https://www.youtube.com/watch?v=hVs8MVydN3A&list=PL4HikwTaYE0GEs7lvlYJQcvKhq0QZGRVn&index=2

LangChain jest biblioteką, która dodaje kolejną warstwę abstrakcji nad API obsługującymi różne modele językowe. Posiada również szereg narzędzi używanych w pracy z LLM takimi jak np. loadery do plików na potrzeby systemu RAG.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

## Quickstart

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm

In [ ]:
response = llm.invoke("Hello")
response

In [ ]:
response.content

In [ ]:
response.response_metadata

In [ ]:
response.response_metadata["model_name"]

**Inne modele**

Dostępne są także inne modele, np.:
- https://pypi.org/project/langchain-anthropic/
- https://pypi.org/project/langchain-ollama/

```python
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic
```

## Parametry `ChatOpenAI`

**`ChatOpenAI`**

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=5
)

llm

**`invoke`**

In [ ]:
llm.invoke("Cześć")

In [ ]:
llm.invoke([{"role": "user", "content": "Say 'this is a test!'"}])

## `batch()` i `stream()`

Funkcji `batch()` używamy w celu wygenerowania odpowiedzi na szereg niezależnych wiadomości.

Funkcji `stream()` używamy kiedy chcemy wygenerować odpowiedź w postaci strumienia tokenów.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

**`batch()`**

Wszystkie prompty przesłane do `batch()` wykonują się równolegle i stanowią osobne wątki konwersacji.

In [ ]:
batch_response = llm.batch(["Say 'this is a test!'", "Who are you?", "What is RAG?"])

In [ ]:
for response in batch_response:
    print(response.content, "\n\n-----\n\n")

---

In [ ]:
batch_response = llm.batch([
    [{"role": "system", "content": "Zamień na wielką literę"}, {"role": "user", "content": "a"}],
    [{"role": "system", "content": "Zamień na wielką literę"}, {"role": "user", "content": "b"}],
    [{"role": "system", "content": "Zamień na wielką literę"}, {"role": "user", "content": "c"}]
])

In [ ]:
for response in batch_response:
    print(response.content, "\n\n-----\n\n")

**`stream()`**

Funkcja `stream()` tworzy generator, po którym możemy iterować. Każdy kolejny token jest generowany przez model dopiero w trakcie iterowania po generatorze.

In [ ]:
llm.stream("Hello")

In [ ]:
import time

In [ ]:
for item in llm.stream("Hello"):
    print(item.content, end='')
    time.sleep(0.4)

> **ZADANIA**

## Prompt template i LLM Chain

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

**`PromptTemplate`**

In [ ]:
template = "Tell me a joke about a {subject}"

In [ ]:
prompt_template = PromptTemplate.from_template(template)

llm = ChatOpenAI(
    model="gpt-4o"
)

In [ ]:
chain = prompt_template | llm
type(chain)

In [ ]:
chain.invoke({"subject": "dog"})

**`ChatPromptTemplate`**

In [ ]:
messages = [
    {"role": "system", "content": "Generate a {separator} separated list of {number} synonyms for the following word"},
    {"role": "user", "content": "{input}"}
]

In [ ]:
chat_prompt_template = ChatPromptTemplate.from_messages(messages)

chain = chat_prompt_template | llm

In [ ]:
chain.invoke({"separator": ";", "number": 3, "input": "chicken"}).content

## Structured output w LangChain

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class ResponseFormat(BaseModel):
    word: str = Field(description="Oryginalne słowo")
    synonyms: list[str] = Field(description="Lista synonimów. Każde słowo powinno być otoczone z obu stron przez **")
    antonyms: list[str] = Field(description="Lista antonimów")

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(ResponseFormat)

In [ ]:
response = llm.invoke("Podaj 3 synonimy i 3 antonimy do 'dobry'")
response

In [ ]:
response.model_dump()

> **ZADANIA**